#### Step 1 导入相关包

In [1]:
import pandas as pd
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModel
import torch

import pickle

from sentence_transformers import SentenceTransformer

d:\anaconda3\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Step 2 加载数据

In [2]:
import os
import pickle

folder = './result/'                # 你的目录名
id2vec = {}                      # 最终的大字典

for fname in tqdm(sorted(os.listdir(folder))):   # 让顺序保持一致
    if not fname.endswith('.pkl'):
        continue
    path = os.path.join(folder, fname)
    with open(path, 'rb') as f:
        part = pickle.load(f)
    id2vec.update(part)          # 把当前分片合并进来

print('合并完成，总条数：', len(id2vec))

100%|██████████| 11/11 [01:48<00:00,  9.83s/it]

合并完成，总条数： 8206415


In [3]:
df_citation = pd.read_csv('./citation.csv',skiprows=range(1,97755011))

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18700\1544796300.py:1: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_citation = pd.read_csv('./citation.csv',skiprows=range(1,97755011))


In [4]:
df_citation

,patent_id,citation_patent_id,citation_category
0,8294741,6299329,cited by other
1,8294741,6326981,cited by other
2,8294741,6327008,cited by examiner
3,8294741,6332030,cited by other
4,8294741,6348929,cited by other
...,...,...,...
49861675,RE50356,9980318,cited by applicant
49861676,RE50356,10164689,cited by examiner
49861677,RE50356,10652955,cited by applicant
49861678,RE50357,8431969,cited by applicant


In [10]:
df_citation['citation_patent_id'] = df_citation['citation_patent_id'].apply(
    lambda x: int(x) if str(x).isdigit() else x
)

#### Step 3 计算相似度

In [5]:
import numpy as npd
from sklearn.metrics.pairwise import cosine_similarity
import os

In [6]:
rows = []

for _, row in tqdm(df_citation.iterrows(),total=len(df_citation)):
    cited = int(row['patent_id']) if str(row['patent_id']).isdigit() else row['patent_id']
    citing = int(row['citation_patent_id']) if str(row['citation_patent_id']).isdigit() else row['citation_patent_id']
    category = row['citation_category']

    if citing not in id2vec or cited not in id2vec:
        continue

    vec_citing = id2vec[citing].reshape(1, -1)
    vec_cited = id2vec[cited].reshape(1, -1)
    sim = cosine_similarity(vec_citing, vec_cited)[0][0]

    rows.append({
        'citing_patent_id': citing,
        'cited_patent_id': cited,
        'similarity': round(float(sim), 3),
        'category': category
    })

result_df = pd.DataFrame(rows)
result_df.to_csv('./result/patent_sim_3.csv', index=False)
print(f"✅ 已保存相似度结果至 patent_sim_3.csv，共 {len(result_df)} 条记录。")

100%|██████████| 49861680/49861680 [1:15:06<00:00, 11063.34it/s]


✅ 已保存相似度结果至 patent_sim_3.csv，共 28386213 条记录。


In [13]:
result_df = pd.DataFrame(rows)
result_df.to_csv('./result/patent_sim_3.csv', index=False)
print(f"✅ 已保存相似度结果至 patent_sim_3.csv，共 {len(result_df)} 条记录。")

✅ 已保存相似度结果至 patent_sim_2.csv，共 44208167 条记录。


In [7]:
rows

[{'citing_patent_id': 8001969,
  'cited_patent_id': 3931976,
  'similarity': 0.443,
  'category': nan},
 {'citing_patent_id': 3987991,
  'cited_patent_id': 3935991,
  'similarity': 0.374,
  'category': nan},
 {'citing_patent_id': 5857301,
  'cited_patent_id': 3943789,
  'similarity': 0.426,
  'category': nan},
 {'citing_patent_id': 3998395,
  'cited_patent_id': 3944004,
  'similarity': 0.486,
  'category': nan},
 {'citing_patent_id': 3989790,
  'cited_patent_id': 3945191,
  'similarity': 0.18,
  'category': nan},
 {'citing_patent_id': 3989367,
  'cited_patent_id': 3945587,
  'similarity': 0.443,
  'category': nan},
 {'citing_patent_id': 3951167,
  'cited_patent_id': 3947279,
  'similarity': 0.297,
  'category': nan},
 {'citing_patent_id': 3952462,
  'cited_patent_id': 3948497,
  'similarity': 0.262,
  'category': nan},
 {'citing_patent_id': 3971499,
  'cited_patent_id': 3951629,
  'similarity': 0.384,
  'category': nan},
 {'citing_patent_id': 3933480,
  'cited_patent_id': 3957743,
  's